# wwd_i — Phases 3-4: train a wake-word head on Colab

Generates per-word data with **ElevenLabs v3**, embeds it through the **frozen `backbone.onnx`** already in the repo, trains a tiny **streaming GRU head**, calibrates the threshold to **FA < 0.5/hr & FR < 5%**, and exports `<word>_head.onnx`.

Change `WAKE_WORD` and re-run to train a different word — the backbone is reused; only the head is retrained.

A **GPU is optional** (the head is tiny; the backbone runs via ONNX Runtime). Set your **`ELEVENLABS_API_KEY`** in step 6.

**Runtime crash-safe:** data and the preprocessed background cache are mirrored to Google Drive and auto-restored (step 5), so a disconnect doesn't cost you the ElevenLabs spend or a multi-GB re-download.

### 1. (optional) GPU

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv || echo 'no GPU — fine, the head is tiny'

### 2. Get the code

`git clone` the repo (push it to GitHub first). For a private repo use `https://<TOKEN>@github.com/<you>/wwd_i.git`.

In [ ]:
import os

REPO_URL = "https://github.com/<you>/wwd_i.git"  # <-- set me
BRANCH = "master"
os.environ["REPO_URL"] = REPO_URL
os.environ["BRANCH"] = BRANCH
!rm -rf /content/wwd_i && git clone --branch "$BRANCH" --depth 1 "$REPO_URL" /content/wwd_i
%cd /content/wwd_i
!git log --oneline -1

### 3. Python 3.14 via uv

In [ ]:
!pip install -q uv
!uv venv --python 3.14
!.venv/bin/python --version

### 4. Install torch + the package + the ElevenLabs SDK

Head training is light; the frozen backbone runs through ONNX Runtime. `--torch-backend=auto` picks a CUDA wheel if a GPU is present, else CPU.

In [ ]:
!uv pip install --no-config --python .venv/bin/python torch torchaudio --torch-backend=auto
!uv pip install --python .venv/bin/python -e . onnxscript elevenlabs

### 5. Google Drive — persist data across runtime crashes

Mounts Drive and fixes a **consistent data dir at `/content/data`** (outside the repo, so re-running the clone cell never wipes it). `restore(name)` pulls an artifact back from Drive if a previous run saved it; `backup(name)` mirrors it out. The expensive things — ElevenLabs clips (cost money) and the preprocessed background cache — are saved once and restored instantly if the runtime dies.

In [ ]:
# --- Google Drive persistence: consistent data dir + restore/backup ---
import os
import shutil
from pathlib import Path

from google.colab import drive

drive.mount("/content/drive")
DATA = "/content/data"  # consistent, OUTSIDE the repo -> survives re-running the clone cell
DRIVE = "/content/drive/MyDrive/wwd_i"  # Drive mirror for the expensive-to-recompute artifacts
os.makedirs(DATA, exist_ok=True)
os.makedirs(DRIVE, exist_ok=True)
os.environ["DATA"], os.environ["DRIVE"] = DATA, DRIVE


def restore(name: str) -> bool:
    """Copy DRIVE/name -> DATA/name if present in Drive. Returns True if restored."""
    src, dst = Path(DRIVE) / name, Path(DATA) / name
    if not src.exists():
        return False
    if src.is_dir():
        shutil.copytree(src, dst, dirs_exist_ok=True)
    else:
        dst.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(src, dst)
    print(f"restored {name} from Drive")
    return True


def backup(name: str) -> None:
    """Mirror DATA/name -> DRIVE/name for crash recovery."""
    src, dst = Path(DATA) / name, Path(DRIVE) / name
    if not src.exists():
        print(f"skip backup: {src} missing")
        return
    if src.is_dir():
        shutil.copytree(src, dst, dirs_exist_ok=True)
    else:
        dst.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(src, dst)
    print(f"backed up {name} to Drive")


print("DATA =", DATA, "| DRIVE =", DRIVE)

### 6. Configure — wake word + API key

The key is read from the environment by the generator; it is never written to disk or code.

In [ ]:
import os

WAKE_WORD = "hey computer"  # <-- the phrase to detect
os.environ["ELEVENLABS_API_KEY"] = ""  # <-- paste your key
os.environ["WAKE_WORD"] = WAKE_WORD
os.environ["WORD_SLUG"] = WAKE_WORD.replace(" ", "_")
assert os.environ["ELEVENLABS_API_KEY"], "set ELEVENLABS_API_KEY above"
print("wake word:", WAKE_WORD)

## Part A — positives & hard negatives (ElevenLabs v3)

### A1. Smoke-test the SDK (one clip)

Validates your key and the SDK surface **before** spending on a batch. If it errors on the API call, the fix is usually a small tweak to `_synthesize`/`_client` in `wwd_i/data/elevenlabs.py` to match your installed SDK version.

In [ ]:
!.venv/bin/python -m wwd_i.data.elevenlabs --phrase "$WAKE_WORD" --out $DATA/smoke --smoke

### A2. Generate positives (~300 diverse clips)

Many voices × prosody settings; cached, so re-running is cheap and bumping `--n-clips` only adds new variants.

In [ ]:
# positives — restore from Drive if a prior run saved them, else generate (paid) and back up
if not restore("pos"):
    !.venv/bin/python -m wwd_i.data.elevenlabs --phrase "$WAKE_WORD" --out $DATA/pos --n-clips 300
    backup("pos")

### A3. Generate hard negatives (near phrases)

The wake phrase's sub-words + generic confusables (`negatives.hard_negative_phrases`) — the main source of false triggers.

In [ ]:
# hard negatives — same restore-or-generate-then-backup pattern
if not restore("hardneg"):
    !.venv/bin/python -m wwd_i.data.elevenlabs --hard-negs-for "$WAKE_WORD" --out $DATA/hardneg --n-clips 20
    backup("hardneg")

## Part B — background noise & music

AudioSet (noise/general) + FMA (music) are decoded **once** into a compact negative-embedding cache in the next section, so training never holds the raw corpus in RAM. These downloads are **skipped automatically if the cache (`bg_neg.npy`) already exists** locally or on Drive — the raw audio is disposable after preprocessing.

### B1. AudioSet — one balanced-train shard

In [ ]:
# Download AudioSet only if the preprocessed cache doesn't already exist (local or Drive).
if os.path.exists(f"{DATA}/bg_neg.npy") or os.path.exists(f"{DRIVE}/bg_neg.npy"):
    print("bg_neg cache present — skipping AudioSet download")
else:
    !mkdir -p $DATA/bg/audioset
    !wget -q -O /tmp/audioset.tar 'https://huggingface.co/datasets/agkphysics/AudioSet/resolve/5a2fa42a1506470d275a47ff8e1fdac5b364e6ef/data/bal_train09.tar?download=true'
    !tar -xf /tmp/audioset.tar -C $DATA/bg/audioset && rm /tmp/audioset.tar
    !echo "audioset files:"; find $DATA/bg/audioset -type f | wc -l

### B2. (optional) FMA music — `fma_small` is ~7.5 GB; skip if bandwidth is tight

In [ ]:
# Optional music corpus (~7.5 GB). Skipped if the cache exists; comment the body out to skip entirely.
if os.path.exists(f"{DATA}/bg_neg.npy") or os.path.exists(f"{DRIVE}/bg_neg.npy"):
    print("bg_neg cache present — skipping FMA download")
else:
    !mkdir -p $DATA/bg/fma
    !wget -q -O /tmp/fma_small.zip https://os.unil.cloud.switch.ch/fma/fma_small.zip
    !unzip -q /tmp/fma_small.zip -d $DATA/bg/fma && rm /tmp/fma_small.zip
    !echo "fma mp3s:"; find $DATA/bg/fma -name '*.mp3' | wc -l

## Part B½ — preprocess backgrounds into a negative-embedding cache (once)

Decodes the raw AudioSet+FMA pool **file-by-file**, crops it, and embeds it through the frozen backbone into a compact `bg_neg.npy` (`[N, W, D]`, ~4 KB/clip) plus a small `noise_pool/` of raw wavs for positive augmentation. This is the fix for the old OOM: training no longer decodes the whole corpus or samples `--n-bg-neg` crops into RAM — it just loads this cache, so the negative count scales freely.

The cache is **word-independent** (frozen backbone), so it's reused for every wake word, and it's backed up to Drive — restored instantly if the runtime dies. Bump `--n-bg-neg` for more negative hours (sharper FA/hr).

In [ ]:
# Build the negative-embedding cache once (or restore it from Drive). Raw $DATA/bg is disposable after this.
r1, r2 = restore("bg_neg.npy"), restore("noise_pool")
if not (r1 and r2):
    !.venv/bin/python -m wwd_i.train.preprocess_bg --background $DATA/bg --out $DATA/bg_neg.npy --n-bg-neg 50000 --noise-pool-dir $DATA/noise_pool
    backup("bg_neg.npy")
    backup("noise_pool")

## Part C — train the head

Embeds the positives/hard-negatives through the frozen backbone, loads the precomputed background negatives from `--bg-neg-emb`, trains the GRU, and calibrates the threshold. Watch the `[gate]` line: **PASS** = FA < 0.5/hr and FR < 5%.

`--background` here points at the small `noise_pool` (additive-noise augmentation for the positives); the bulk negatives come from the cache, so there's no `--n-bg-neg`/`--max-bg` decode at train time. The trained head is mirrored to Drive.

In [ ]:
!mkdir -p $DATA/artifacts
!.venv/bin/python -m wwd_i.train.train_head --word "$WAKE_WORD" --positives $DATA/pos --hard-neg $DATA/hardneg --background $DATA/noise_pool --bg-neg-emb $DATA/bg_neg.npy --n-aug 5 --epochs 40 --hidden 48 --out $DATA/artifacts/${WORD_SLUG}_head.onnx --threshold-out $DATA/artifacts/${WORD_SLUG}_head.json
backup("artifacts")

## Part D — download the head + calibration

In [ ]:
from google.colab import files

slug = os.environ["WORD_SLUG"]
files.download(f"{DATA}/artifacts/{slug}_head.onnx")
files.download(f"{DATA}/artifacts/{slug}_head.json")

## Interpreting the gate

- **PASS** (FA < 0.5/hr, FR < 5%): drop `<word>_head.onnx` + its `.json` (threshold + refractory) into the repo — the frozen backbone plus this head are a complete detector → Phase 5 (streaming runtime).
- **High FA**: more / harder negatives — rebuild the cache with a bigger preprocess `--n-bg-neg` (and more background files), or add more hard-neg phrases.
- **High FR**: more positives (`--n-clips`) or augmentation variety (`--n-aug`); confirm the A1/A2 clips are on-phrase.
- FA/hr is only as fine as the negative hours — scale the preprocess `--n-bg-neg` to actually resolve < 0.5/hr.